# Advancing BERT: NLP Programming Assignment 2 (Part 3)
**Objective**: Explore and improve key limitations of BERT-based models by assessing Tokenization gaps. Specifically, evaluating WordPiece (baseline) vs. BPE vs. Character-level tokenization for Question Answering on a SQuAD subset.

This notebook is structured to run end-to-end and will output metrics for your final discussion.

In [ ]:
!pip install transformers datasets tokenizers evaluate accelerate pandas torch


In [ ]:
import time
import os
import torch
import json
import pandas as pd
import evaluate
from datasets import load_dataset
from tokenizers import ByteLevelBPETokenizer
from tokenizers import Tokenizer, models, pre_tokenizers
from transformers import (
    BertTokenizerFast,
    PreTrainedTokenizerFast,
    BertForQuestionAnswering,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)


In [ ]:
print("Loading SQuAD subset...")
dataset = load_dataset("squad")

# We use sub-sets for faster turnarounds: 2000 train, 500 val
# For char-level tokenization which is very expensive, we will use smaller subsets inside its block.
train_dataset = dataset["train"].select(range(2000))
val_dataset = dataset["validation"].select(range(500))

print(train_dataset[0])


## Experiment 1: Baseline (WordPiece Tokenization)
Standard BERT uses WordPiece tokenization.


In [ ]:
print("=== Starting Baseline Experiment (WordPiece) ===")
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

def preprocess(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=384,
        truncation="only_second",
        padding="max_length",
        return_offsets_mapping=True
    )

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(inputs["offset_mapping"]):
        answer = examples["answers"][i]
        if len(answer["answer_start"]) == 0:
            start_positions.append(0)
            end_positions.append(0)
            continue
            
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])

        sequence_ids = inputs.sequence_ids(i)

        idx = 0
        while idx < len(sequence_ids) and sequence_ids[idx] != 1:
            idx += 1
        context_start = idx

        while idx < len(sequence_ids) and sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        if context_start >= len(offsets) or offsets[context_start][0] > end_char or offsets[context_end][1] < start_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offsets[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx-1)

            idx = context_end
            while idx >= context_start and offsets[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx+1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    inputs.pop("offset_mapping")
    return inputs

train_baseline = train_dataset.map(preprocess, batched=True, remove_columns=train_dataset.column_names)
val_baseline = val_dataset.map(preprocess, batched=True, remove_columns=val_dataset.column_names)

model = BertForQuestionAnswering.from_pretrained("bert-base-uncased")

training_args = TrainingArguments(
    output_dir="./results_baseline",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="no"
)

data_collator = DataCollatorWithPadding(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_baseline,
    eval_dataset=val_baseline,
    processing_class=tokenizer,
    data_collator=data_collator
)

start_time = time.time()
trainer.train()
baseline_time = time.time() - start_time
print("Baseline Training Time:", baseline_time)

eval_metrics_baseline = trainer.evaluate()
print("Baseline Evaluation Complete:", eval_metrics_baseline)

model.save_pretrained("bert_baseline")
size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, dn, filenames in os.walk("bert_baseline")
    for f in filenames
)
baseline_size = size / 1e6
print("Baseline Model Size (MB):", baseline_size)


## Experiment 2: BPE Tokenization
We learn common subword units by merging frequent character pairs.


In [ ]:
print("=== Starting BPE Tokenizer Experiment ===")

texts = []
for item in dataset["train"].select(range(5000)):
    texts.append(item["context"])
    texts.append(item["question"])

os.makedirs("bpe_tokenizer", exist_ok=True)
with open("text.txt", "w", encoding="utf-8") as f:
    for t in texts:
        f.write(t + "\n")

bpe_tokenizer_train = ByteLevelBPETokenizer()
bpe_tokenizer_train.train(
    files=["text.txt"],
    vocab_size=30000,
    min_frequency=2
)
bpe_tokenizer_train.save_model("bpe_tokenizer")

bpe_tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=None,
    vocab_file="bpe_tokenizer/vocab.json",
    merges_file="bpe_tokenizer/merges.txt",
    unk_token="[UNK]",
    pad_token="[PAD]",
    cls_token="[CLS]",
    sep_token="[SEP]",
    mask_token="[MASK]"
)

def preprocess_bpe(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = bpe_tokenizer(
        questions,
        examples["context"],
        max_length=384,
        truncation="only_second",
        padding="max_length",
        return_offsets_mapping=True
    )

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(inputs["offset_mapping"]):
        answer = examples["answers"][i]
        if len(answer["answer_start"]) == 0:
            start_positions.append(0)
            end_positions.append(0)
            continue
            
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])

        sequence_ids = inputs.sequence_ids(i)

        idx = 0
        while idx < len(sequence_ids) and sequence_ids[idx] != 1:
            idx += 1
        context_start = idx

        while idx < len(sequence_ids) and sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        if context_start >= len(offsets) or context_end < 0 or offsets[context_start][0] > end_char or offsets[context_end][1] < start_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offsets[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx-1)

            idx = context_end
            while idx >= context_start and offsets[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx+1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    if "offset_mapping" in inputs:
        inputs.pop("offset_mapping")
    return inputs

train_bpe = train_dataset.map(preprocess_bpe, batched=True, remove_columns=train_dataset.column_names)
val_bpe = val_dataset.map(preprocess_bpe, batched=True, remove_columns=val_dataset.column_names)

model_bpe = BertForQuestionAnswering.from_pretrained("bert-base-uncased")
model_bpe.resize_token_embeddings(len(bpe_tokenizer))

training_args_bpe = TrainingArguments(
    output_dir="./results_bpe",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="no"
)

data_collator_bpe = DataCollatorWithPadding(bpe_tokenizer)

trainer_bpe = Trainer(
    model=model_bpe,
    args=training_args_bpe,
    train_dataset=train_bpe,
    eval_dataset=val_bpe,
    processing_class=bpe_tokenizer,
    data_collator=data_collator_bpe
)

start_time = time.time()
trainer_bpe.train()
bpe_time = time.time() - start_time
print("BPE Training Time:", bpe_time)

eval_metrics_bpe = trainer_bpe.evaluate()

model_bpe.save_pretrained("bert_bpe")
size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, dn, filenames in os.walk("bert_bpe")
    for f in filenames
)
bpe_size = size / 1e6
print("BPE Model Size (MB):", bpe_size)


## Experiment 3: Character-level Tokenization
Instead of breaking into words or subwords, we break into distinct characters. We will evaluate Char-level tokenizer on a smaller SQuAD subset because the sequence length becomes extremely high (approx 4-5x longer vs subword approaches).


In [ ]:
print("=== Starting Character-Level Tokenizer Experiment ===")

train_dataset_char = dataset["train"].select(range(500))
val_dataset_char = dataset["validation"].select(range(100))

chars = list("abcdefghijklmnopqrstuvwxyz0123456789.,!?;:-'\"() ")
vocab = {c: i+5 for i, c in enumerate(chars)}
vocab["[PAD]"] = 0
vocab["[UNK]"] = 1
vocab["[CLS]"] = 2
vocab["[SEP]"] = 3
vocab["[MASK]"] = 4

os.makedirs("char_tokenizer", exist_ok=True)
with open("char_tokenizer/vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f)

base_tokenizer = Tokenizer(models.BPE(vocab=vocab, merges=[]))
base_tokenizer.pre_tokenizer = pre_tokenizers.Split(pattern="", behavior="isolated")
base_tokenizer.save("char_tokenizer/tokenizer.json")

char_tokenizer = PreTrainedTokenizerFast(
    tokenizer_file="char_tokenizer/tokenizer.json",
    unk_token="[UNK]",
    pad_token="[PAD]",
    cls_token="[CLS]",
    sep_token="[SEP]",
    mask_token="[MASK]"
)

def preprocess_char(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = char_tokenizer(
        questions,
        examples["context"],
        max_length=512, 
        truncation="only_second",
        padding="max_length",
        return_offsets_mapping=True
    )

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(inputs["offset_mapping"]):
        answer = examples["answers"][i]
        if len(answer["answer_start"]) == 0:
            start_positions.append(0)
            end_positions.append(0)
            continue
            
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])

        sequence_ids = inputs.sequence_ids(i)
        idx = 0
        while idx < len(sequence_ids) and sequence_ids[idx] != 1:
            idx += 1
        context_start = idx

        while idx < len(sequence_ids) and sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        if context_start >= len(offsets) or context_end < 0 or offsets[context_start][0] > end_char or offsets[context_end][1] < start_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offsets[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx-1)

            idx = context_end
            while idx >= context_start and offsets[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx+1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    if "offset_mapping" in inputs:
        inputs.pop("offset_mapping")
    return inputs

train_char_mapped = train_dataset_char.map(preprocess_char, batched=True, remove_columns=train_dataset_char.column_names)
val_char_mapped = val_dataset_char.map(preprocess_char, batched=True, remove_columns=val_dataset_char.column_names)

model_char = BertForQuestionAnswering.from_pretrained("bert-base-uncased")
model_char.resize_token_embeddings(len(char_tokenizer))

training_args_char = TrainingArguments(
    output_dir="./results_char",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="no"
)

data_collator_char = DataCollatorWithPadding(char_tokenizer)

trainer_char = Trainer(
    model=model_char,
    args=training_args_char,
    train_dataset=train_char_mapped,
    eval_dataset=val_char_mapped,
    processing_class=char_tokenizer,
    data_collator=data_collator_char
)

start_time = time.time()
trainer_char.train()
char_time = time.time() - start_time
print("Char Training Time:", char_time)

eval_metrics_char = trainer_char.evaluate()

model_char.save_pretrained("bert_char")
size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, dn, filenames in os.walk("bert_char")
    for f in filenames
)
char_size = size / 1e6
print("Char Model Size (MB):", char_size)


## Evaluate all Results
We compile the extracted metrics (eval loss, training time, model size) over our subset tests.


In [ ]:
results = pd.DataFrame({
    "Tokenizer": ["WordPiece", "BPE", "Character"],
    "Eval Loss": [eval_metrics_baseline.get("eval_loss", 0.0), eval_metrics_bpe.get("eval_loss", 0.0), eval_metrics_char.get("eval_loss", 0.0)],
    "Training Time (s)": [baseline_time, bpe_time, char_time],
    "Model Size (MB)": [baseline_size, bpe_size, char_size]
})

print("\n\n" + "="*50)
print("FINAL EXPERIMENT RESULTS")
print("="*50)
print(results.to_string(index=False))

results.to_csv("tokenization_results.csv", index=False)
